# Youtube Music Playlist API Notebook

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)

from ytmusic_library import YTMusicPlaylists

RUN_API_AUTH_TEST = True
HEADER_FILE = '../browser.json'
PLAYLIST_TSV_DIR = '../playlists/'

In [2]:
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

Using header file: ../browser.json
-> Method: Browser Authentication (cookies). Skipping OAuth client init.
Test Passed in 2.28 seconds
Using ytmusicapi version: 1.11.1
Loaded 504 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [3]:
playlist_names = [
# 'Indie Folk radio', 
# 'xr Exotica radio',
# 'xr nudisco radio',
# 'folk acoustic radio', 
# 'xr neopsychedelia radio',
# 'xr indierock radio',
# 'xr chillwave radio', 
# 'electronic Underjord radio',
# 'electronic Focus radio',
'electronic post dub radio',
# "Reggae radio",
# "xr indie radio",
# "xr IndieFolk radio",
# "xr shoegaze radio",
# "xr chillmusic radio", # liminel
# 'xr treemusic radio',
# 'rock surf radio',
# 'reggae classic radio',
# 'nudisco radio',
# 'xr Elephant6 radio',
# 'future beats radio',
# 'xr desertblues radio'
# 'indie krill vibe radio'

]
MIN_RADIO_LIKE_TO_SPLIT=2
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_radio_playlist(pl_info, 
        move_like=True, create_like_playlist=True, remove_dislike=True, remove_not_like=True,
        min_num_like=MIN_RADIO_LIKE_TO_SPLIT,  sleep=1, verbose=True,
    )


Playlist electronic post dub radio counters: {'moved_like': 2}


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [8]:
playlist_names = [
  # 'electronic New Wave old and new radio',
  # 'Reggae radio',
  # 'helvetia duster static cult'
  # 'electronic New Wave old and new radio',
  # 'folk acoustic radio',
  'xr Elephant6 radio',
    # 'xr treemusic radio' #'jazz fusion cross radio', 'soul groove instrumental radio', 'jazz funk cortex vibe radio'
]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=False)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


Loaded 469207 playounts from 176023 tracks
Created sorted pl: xr Elephant6 radio [01-17-2026] PLWptjpDqazOyCkLsGcBRaF_2BXLgNbZdB, and  deleted original pl: PLWptjpDqazOyddeiqM76mQJPMC57rRPH9


## Save playlist backup tsv

In [ ]:
PLAYLIST_NAME = 'zz not like 0' # Liked Music
USE_CACHE = False
_pl_id = Y.query_by_title(PLAYLIST_NAME).playlistId
_pl_tracks, _pl_metadata = Y.save_playlist_tsv(Y.playlist_get_info(_pl_id, use_cache=USE_CACHE))

                                                                  0
owned                                                          True
id                               PLWptjpDqazOyCkLsGcBRaF_2BXLgNbZdB
privacy                                                     PRIVATE
description       generated sorting by lastfm playcount for xr E...
views                                                          None
duration                                                   5+ hours
trackCount                                                      268
title                                            xr Elephant6 radio
author            {'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFUM...
year                                                           2026
related                                                          []
duration_seconds                                              55414


## Update _not_like tsv

In [37]:
not_like_tracks = Y.collect_all_not_like_tracks_from_tsvs()
not_like_tracks.to_csv(Y.not_like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to not_like tsv: {Y.not_like_tsv}')

Found 4 not like playlists out of the 531 total
Updated not liked tracks, contains 10121 entries.
Saved 10121 entries to not_like tsv: ../playlists/_not_liked_tracks.tsv


## Update _like tsv and print like coverage

In [ ]:
like_tracks = Y.collect_all_like_tracks_from_tsvs()
like_tracks.to_csv(Y.like_tsv, sep='\t', header=True)
print(f'Saved {len(like_tracks)} entries to like tsv: {Y.like_tsv}')
# run 1-2026, Found 260 like playlists out of the 527 total


Found 260 like playlists out of the 527 total
Beats Lofi.tsv	92.2% currently liked (of 141 total tracks)
Bossa Nova.tsv	88.0% currently liked (of 25 total tracks)
Brass n chill.tsv	93.0% currently liked (of 258 total tracks)
Chillwave.tsv	97.0% currently liked (of 367 total tracks)
Folk.tsv	95.5% currently liked (of 441 total tracks)
Grunge.tsv	99.0% currently liked (of 103 total tracks)
Hip Hop 1980s.tsv	96.0% currently liked (of 25 total tracks)
Hip Hop 1990s east coast.tsv	100.0% currently liked (of 29 total tracks)
Hip Hop 1990s west Coast.tsv	100.0% currently liked (of 74 total tracks)
Hip Hop 1990s.tsv	96.4% currently liked (of 801 total tracks)
Hip Hop 2000s southern.tsv	84.0% currently liked (of 25 total tracks)
Hip Hop 2000s.tsv	96.7% currently liked (of 92 total tracks)
Hip Hop Hits.tsv	97.2% currently liked (of 250 total tracks)
Indie 1990s Rock.tsv	98.7% currently liked (of 76 total tracks)
Indie 2000s.tsv	97.9% currently liked (of 233 total tracks)
Indie psych.tsv	94.4% cu

## Get Public Playlists

In [4]:
skip_if_contains = (' radio', 'ym ', ' albums', 'zz not', 'zzz ', 'yyz ', 'Episodes for ', 'Liked Music', 'Podcast Queue')
df_public = Y.get_playlists_by_privacy(privacy='PUBLIC', skip_if_contains=skip_if_contains)
print(f'\nFound {len(df_public)} public playlists')
df_public[['title', 'views', 'trackCount', 'duration']].head(30)

# update function run 2-2026, 270 public playlists
# run 1-2026, 270 public playlists
# run 8-2025, 267 public playlists
# run 1-2025, 270 public playlists, fine descriptions

Found public: 'electronic post dub', Stats: 185 tracks, 6 views, 14+ hours, Description: favorites... | [PLWptjpDqazOxmeskGJ9l3z5pMhMP39TNx]
Found public: 'Reggae', Stats: 530 tracks, 7 views, 35+ hours, Description: None... | [PLWptjpDqazOy1OO5b_SLn4r5Rx3XxemYo]
Found public: 'good', Stats: 85 tracks, 37 views, 20+ hours, Description: None... | [PLDLIpMUUSUXETeVoqYD4YcNKUBHc1g6lA]
Found public: 'xr RootsMusic', Stats: 92 tracks, None views, 5 hours, 53 minutes, Description: favorites compiled from reddit sub r/Roo... | [PLWptjpDqazOzLoVTY4_ghLS-LXZGY4TEE]
Found public: 'blues', Stats: 879 tracks, 2 views, 56+ hours, Description: None... | [PLWptjpDqazOywwzZn-tkczbidWrMVG8_o]
Found public: 'xr acidhouse', Stats: 31 tracks, None views, 3 hours, 1 minute, Description: favorites compiled from reddit sub r/aci... | [PLWptjpDqazOxMwX0OKw8q1P9qp5GOXOhQ]
Found public: 'xr Rock', Stats: 322 tracks, 7 views, 26+ hours, Description: favorites compiled from reddit sub r/Roc... | [PLWptjpDqazOwNTc

,title,views,trackCount,duration
155,trip hop & Ambien®,700,220.0,17+ hours
166,indie krill vibe,579,451.0,30+ hours
18,rock 1970s classic,575,1079.0,69+ hours
191,rock 1960s classic,527,923.0,55+ hours
143,jazz spiritual,256,31.0,"3 hours, 29 minutes"
10,xr 60sMusic,249,750.0,46+ hours
222,hiphop indie,185,31.0,"2 hours, 1 minute"
37,reggae classic,177,212.0,12+ hours
225,Rock 1967-1969,144,197.0,13+ hours
93,blues deep roots,141,268.0,14+ hours


## Get Playlist Counts

## Get radio playlists with lots of likes

In [5]:
%%time
playlist_file = os.path.join(PLAYLIST_TSV_DIR, '_playlist_radio_counts.tsv')
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
playlists.to_csv(playlist_file, sep='\t', index=False)
# Note: now part of ytmusic backup

playlists.head(20)
# last run 2-2026

Getting playlist track count for playlists (takes ~5 minutes)
Only for playlists with radio in the title
CPU times: total: 34.8 s
Wall time: 4min 25s


,title,track_count,privacy,playlist_id
0,indie krill vibe radio,22,PRIVATE,PLWptjpDqazOy10kXns-iHXWsxBRQyb_Si
78,xr Samplehunters radio,23,PRIVATE,PLWptjpDqazOwxUqgcJItuO9jNl3Jcg_fL
73,xr foreignrap radio,25,PRIVATE,PLWptjpDqazOzjYZVOFq_e_A3nlSdFe8vI
92,xr dubtechno radio,25,PRIVATE,PLWptjpDqazOxlekzaH0s1kBmoDYwfd_to
88,xr gamemusic radio,27,PRIVATE,PLWptjpDqazOyYqtWSh1_cyv_IDFvs8KS7
129,xr chillmusic radio,29,PRIVATE,PLWptjpDqazOxNK0_Dt9ijVP9RJZMM_Gd9
85,xr icm radio,30,PRIVATE,PLWptjpDqazOwIeSpwFTzviniXIDxo7tIL
98,xr BinauralMusic radio,31,PRIVATE,PLWptjpDqazOwg0uL9JgZpUWJK58HlV7_E
19,xr GypsyJazz radio,32,PRIVATE,PLWptjpDqazOx_aDtCHfgLIo0att6y0xCb
89,xr frisson radio,43,PRIVATE,PLWptjpDqazOzZWeXU9wroauyqrzaP5Os4


# One Off

### Check albums_musicbee_like_from_ytmusic_matches

1. Looks at `albums_musicbee_like_from_ytmusic_matches.m3u` (copied in musicbe in db format saved as `./logs/maybe_like.tsv`)
2. Checks ytmusic to see if actual like
3. Output pasted in gemini can look to see if any bad like matches or misses (not like)
4. Goal is to batch like or not like `albums_musicbee_like_from_ytmusic_matches.m3u`

In [20]:
import pandas as pd
import re
import difflib
import sys
import os
from collections import defaultdict

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Source info: Created 2/2026 using musicbee db copy to tsv logic

INPUT_DIR = './logs/'
OUTPUT_DIR = './logs/'
INPUT_FILE = os.path.join(INPUT_DIR, 'maybe_like.tsv')

M3U_LIKE = os.path.join(OUTPUT_DIR, 'artists_musicbee_IS_LIKE_from_ytmusic_matches.m3u')
M3U_NOT_LIKE = os.path.join(OUTPUT_DIR, 'artists_musicbee_IS_NOT_LIKE_from_ytmusic_matches.m3u')

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

# Keywords that MUST match if present in one but not the other
CRITICAL_KEYWORDS = {'demo', 'live', 'instrumental', 'remix', 'acoustic', 'session', 'reprise', 'part'}

# HARDCODED OVERRIDES
# Format: ('normalized_artist', 'normalized_title'): True (Like) / False (Not Like)
MANUAL_OVERRIDES = {
    # User Request: Duke Ellington -> Like
    ('dukeellingtonandhisorchestra', 'diminuendoinblueandcrescendoinblue'): True,
    ('dukeellington', 'diminuendoinblueandcrescendoinblue'): True, # cover both artist variants
    
    # User Request: Durand Jones (Demo) -> Not Like
    ('durandjones', 'dontyouknowdemo'): False,
    ('durandjonesandtheindications', 'dontyouknowdemo'): False,

    # User Request: J Dilla -> Not Like (Fix false ID match)
    ('jdilla', 'wontdo'): False,

    # User Request: Young Marble Giants (Clicktalk) -> Not Like (Fix false positive to "The Clock")
    ('youngmarblegiants', 'clicktalk'): False,
    
    # Previous Fixes
    ('thecosmicjokers', 'galacticsupermarketpart3'): False,
    ('outkast', 'interlude'): False,
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def normalize(s):
    """Standardizes string: lowercase, remove musicbee tags, and strip specific noise suffixes."""
    s = str(s).lower()
    s = re.sub(r'\[.*?\]', '', s) 
    
    # NEW: Remove specific long administrative suffixes that confuse fuzzy matchers
    # Example: "Clicktalk (Taken From The Final Day Single)" -> "Clicktalk"
    s = re.sub(r'\(taken from.*?\)', '', s)
    s = re.sub(r'\(single version\)', '', s)
    
    # Keep alphanumeric and spaces
    s = re.sub(r'[^a-z0-9 ]', '', s)
    return s.strip()

def normalize_key(s):
    """Stricter normalization for dictionary keys (no spaces)."""
    return re.sub(r'\W+', '', normalize(s))

def get_keywords(s):
    """Extracts critical version keywords."""
    s = normalize(s)
    found = set()
    for kw in CRITICAL_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', s):
            found.add(kw)
    return found

def check_version_mismatch(in_title, res_title):
    in_kws = get_keywords(in_title)
    res_kws = get_keywords(res_title)
    if in_kws != res_kws:
        return True, f"Version mismatch: {in_kws} vs {res_kws}"
    return False, ""

def get_enhanced_score(s1, s2):
    n1, n2 = normalize(s1), normalize(s2)
    if not n1 or not n2: return 0
    
    base_score = int(difflib.SequenceMatcher(None, n1, n2).ratio() * 100)
    
    # Substring Boost (Only if substantial length)
    if (n1 in n2 or n2 in n1) and len(min(n1, n2, key=len)) > 3:
        return max(base_score, 95)
    return base_score

def save_m3u(filepath, paths):
    clean = [str(p) for p in paths if p and str(p).strip() and str(p).lower() != 'nan']
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("#EXTM3U\n")
        f.write("# Generated using musicbee db copy to tsv logic (2/2026)\n")
        f.write('\n'.join(clean) + '\n')
    print(f"Saved {filepath} ({len(clean)} tracks)")

# ==========================================
# 3. LOAD DATA
# ==========================================
print("--- LOADING DATA ---")
liked_ids, liked_lookup = set(), defaultdict(list)

# Load Liked DB
if hasattr(Y, 'like_tsv') and os.path.exists(Y.like_tsv):
    try:
        df_db = pd.read_csv(Y.like_tsv, sep='\t')
        id_col = next((c for c in ['videoId', 'id'] if c in df_db.columns), 'Unnamed: 0')
        liked_ids = set(df_db[id_col].dropna().astype(str))
        
        for _, row in df_db[['title', 'artist']].fillna('').iterrows():
            if row['title']: 
                norm_key = normalize_key(row['title'])
                liked_lookup[norm_key].append(row.to_dict())
        print(f"Loaded {len(liked_ids)} Liked IDs.")
    except Exception as e:
        print(f"Error reading Liked DB: {e}")

# Load Input
if os.path.exists(INPUT_FILE):
    df_input = pd.read_csv(INPUT_FILE, sep='\t').fillna('')
    df_input.columns = df_input.columns.str.strip()
    print(f"Loaded {len(df_input)} items.")
else:
    print(f"ERROR: Input file not found.")
    df_input = pd.DataFrame()

# ==========================================
# 4. PROCESSING LOOP
# ==========================================
results, m3u_liked, m3u_not_liked = [], [], []
total = len(df_input)

if total > 0:
    print(f"Processing {total} items...")
    
    for i, row in df_input.iterrows():
        if i % 10 == 0: sys.stdout.write(f"\rProgress: {int((i+1)/total*100)}%"); sys.stdout.flush()

        artist = str(row.get('Artist', '')).strip()
        title = str(row.get('Title', '')).strip()
        album = str(row.get('Album', '')).strip()
        path = str(row.get('Path', '')).strip()
        
        clean_title = re.sub(r'\[.*?\]', '', title).strip()
        query = f"{clean_title} {artist}" if artist else f"{clean_title} {album}"

        # Manual Override Check
        override_key = (normalize_key(artist), normalize_key(clean_title))
        override_val = MANUAL_OVERRIDES.get(override_key)

        matches = []
        for item in Y.yt.search(query, filter='songs')[:5]:
            is_liked, reason = False, ""
            
            # A. ID Check
            if item['videoId'] in liked_ids:
                is_liked, reason = True, "ID Match"
            else:
                # B. Fuzzy DB Check
                res_norm = normalize_key(item['title'])
                cands = liked_lookup.get(res_norm, [])
                if not cands:
                    close_keys = difflib.get_close_matches(res_norm, liked_lookup.keys(), n=1, cutoff=0.85)
                    for k in close_keys: cands.extend(liked_lookup[k])
                
                res_arts = [normalize_key(a['name']) for a in item.get('artists', [])]
                for c in cands:
                    db_art = normalize_key(c['artist'])
                    if any(ra in db_art or db_art in ra for ra in res_arts):
                        is_liked, reason = True, f"Ref: {c['title']}"
                        break
            
            score_t = get_enhanced_score(clean_title, item['title'])
            item_art = " ".join([a['name'] for a in item.get('artists', [])])
            score_a = get_enhanced_score(artist, item_art)
            final_score = int(score_t * 0.7 + score_a * 0.3) if artist else score_t
            
            is_mismatch, mismatch_note = check_version_mismatch(clean_title, item['title'])
            
            # C. ID Sanity Check (New Fix)
            # If ID Matches but Text Score is abysmal (<40), it's likely a bad metadata link or sample ID match
            if is_liked and reason == "ID Match" and score_t < 40:
                  is_mismatch = True
                  mismatch_note = f"Suspicious ID Match (Score {score_t})"

            matches.append({
                'data': item, 'is_liked': is_liked, 'reason': reason,
                'score': final_score, 'title_score': score_t,
                'is_mismatch': is_mismatch, 'mismatch_note': mismatch_note
            })


        # Selection
        matches.sort(key=lambda x: x['score'], reverse=True)
        best = None
        status = "Not Liked"
        notes = ""

        if override_val is not None:
            status = "LIKED" if override_val else "Not Liked"
            notes = f"Manual Override: FORCE {str(override_val).upper()}"
            (m3u_liked if override_val else m3u_not_liked).append(path)
            best = matches[0] if matches else None
        else:
            valid_liked = next((m for m in matches if m['is_liked'] and not m['is_mismatch'] and m['title_score'] > 65), None)
            
            if valid_liked:
                best = valid_liked
                status = "LIKED"
                notes = best['reason']
                m3u_liked.append(path)
            else:
                best = matches[0] if matches else None
                status = "Not Liked"
                m3u_not_liked.append(path)
                if best and best['is_liked']:
                     notes = f"Rejected: {best['mismatch_note']}" if best['is_mismatch'] else f"Rejected: Low Score ({best['title_score']})"
                elif not best: notes = "No Results"
                else: notes = "Closest match not in Liked DB"

        if best:
             results.append({
                'Status': status, 'Score': best['score'], 'TitleScore': best['title_score'],
                'Input Query': query[:35], 'Found Title': best['data']['title'][:35], 'Match Notes': notes
            })
        else:
             results.append({'Status': status, 'Score':0, 'TitleScore':0, 'Input Query': query[:35], 'Found Title': '', 'Match Notes': notes})

    print("\nDone.")
    save_m3u(M3U_LIKE, m3u_liked)
    save_m3u(M3U_NOT_LIKE, m3u_not_liked)

    if results:
        df_out = pd.DataFrame(results).fillna('')
        print("\n" + "="*80 + "\nMATCHING RESULTS\n" + "="*80)
        def sort_helper(row):
            if row['Status'] == 'LIKED': return 0
            if 'Rejected' in str(row['Match Notes']): return 1
            if 'Manual Override' in str(row['Match Notes']): return 0.5
            return 2
        df_out['sort_key'] = df_out.apply(sort_helper, axis=1)
        df_out.sort_values(by=['sort_key', 'Score'], ascending=[True, False], inplace=True)
        print(df_out.drop(columns=['sort_key']).to_string())

--- LOADING DATA ---
Loaded 38329 Liked IDs.
Loaded 4 items.
Processing 4 items...
Progress: 25%
Done.
Saved ./logs/artists_musicbee_IS_LIKE_from_ytmusic_matches.m3u (0 tracks)
Saved ./logs/artists_musicbee_IS_NOT_LIKE_from_ytmusic_matches.m3u (4 tracks)

MATCHING RESULTS
      Status  Score  TitleScore                          Input Query Found Title Match Notes
0  Not Liked      0           0  Her Eyelids Say Félicia Atkinson &               No Results
1  Not Liked      0           0  All Night I Carpenter Félicia Atkin              No Results
2  Not Liked      0           0      Let Him Run Wild The Beach Boys              No Results
3  Not Liked      0           0           The Hard One The Beta Band              No Results


### Blues list

In [ ]:
for title in Y.playlists['title'].unique():
  if "blues" in title.lower():
    print(title) 

blues
blues chicago
blues delta roots
blues electric
blues texas roots
rock blues
xr blues radio
xr bluesrock
xr bluesrock albums
xr bluesrock radio


In [ ]:
pl_info = Y.playlist_get_info(Y.query_by_title('blues texas roots').playlistId)
for track in pl_info['tracks']:
  print()
  print(f"Title: {track['title']}\nArtist: "
        f"{track['artists'][0]['name'] if track['artists'] else 'Unknown'}\n" 
        f"Album: {track['album']['name'] if track['album'] else 'Unknown'}\n"
        f"VideoID: xx\n"
  )

Title: Mean Old World
Artist: T-Bone Walker
Album: T-Bone Blues
VideoID: M_ZDGFBKbBM

Title: Groundhog Blues
Artist: Lil' Son Jackson
Album: Blues Come to Texas
VideoID: i0yihJQ27yg

Title: Captain, Captain!
Artist: Mance Lipscomb
Album: Captain, Captain!
VideoID: -Wp_0pAf4hM

Title: Ticket Agent
Artist: Lil' Son Jackson
Album: Blues Come to Texas
VideoID: ZHWTbE2ZN6U

Title: Back to Santa Fe
Artist: Lee Hunter
Album: Texas Blues: Bill Quinn's Gold Star Recordings
VideoID: FccZzqOMh4A

Title: Freddie
Artist: Mance Lipscomb
Album: The Best of Mance Lipscomb
VideoID: 90vatUNRvBk

Title: Strike Blues
Artist: L.C. Williams
Album: Texas Blues: Bill Quinn's Gold Star Recordings
VideoID: zO_5gdnFua0

Title: Bye Bye Baby Blues
Artist: Little Hat Jones
Album: Don't Leave Me Here: The Blues Of Texas, Arkansas & Louisiana (1927-1932)
VideoID: KI4hlzHUvZM

Title: I Don't Want No Woman
Artist: L.C. Lightnin Jr. Williams
Album: The Mercury Blues Story (1945-1955) - Southwest Blues, Vol. 2
VideoID: 3

In [ ]:
track_ids = [
    "dUKcpeVvHsU",
    "FKCwRDB42wo",
    "s5Y8o6PKFnc",
    "6rf8QfII7kg",
    "AzRuGcEcaVo",
    "EOkIazVPwEo",
    "vQClDmtYBys",
    "ac8rL2Ga7pA",
    "PsrFAbFdNWI",
    "jviYDmdLX2A",
    "ayuhW1p_7wo",
    "oYGJVCOklOo",
    "6YFuk97BdPw",
    "xxoO9Da5_Dg",
    "I4ZLZag9GFs",
    "FCvHEXdtPl4",
    "ZG6xibft3pA",
    "1RKlbgvY2gA",
    "r72I3Ps5zYc",
    "UwXA62rDK7U",
    "tMd7qNr5Ct8",
    "Ex0uCvA3tEM",
    "rB0pXI0Fna4",
    "KI4hlzHUvZM",
    "Jl5W8RaG8GM",
    "Uqcu9wZnGL4",
    "_NPUfBpr0RE",
    "2v0yPrdgfRQ",
    "IPcPYsJwJZc",
    # "qS3gCsJ0",
    # "I91a6_-6M",
    "02IkbJkazLs",
    "LfgK9FckrWo",
    "1tE36dBdsP0",
    "Px6unLbNu4Q",
    "05ypnm_Wcls",
    "Wp1hKpIKzNE",
    "CUIF8cGf3ow",
    "aANSKrs87FA",
    "WmJhMFZEzc8",
    "HP8b6B9-wNg",
    "vn28G8yLsw",
    "TjazJ39k1GY",
    "71Ti_TdpJHE",
    "tZRve-Iehsk",
    "qasPCNQuo88",
    "o8l4LTBwoBE",
    "Ftp4TZKGoug",
    "sk4Js6Q58h8",
    "a5zslcqNCKo",
    "RyZm921d1BU",
    "u-FvKRUTIq4",
    "itiEgttbO0M",
    "-b2XeTb7dY4",
    "mhIQYEoLrjU",
    "AIAw2fAFNMA",
    "Ee7Vg3kf0yY",
    "5SyV43rP-xA",
    "rmJRtSGk-lM",
    "Egfcm2ldXU4",
    "h8R3Oy28BKM",
    "APSCifra1eY",
    "_0gCaSaoULw",
    "_rCswNoJnLI",
    "2seZr4Fl4XA",
    "gIHoUxzRoL8",
    "m09tdbsnxKc",
    "nBGloVKAUyU",
    "2jyGShRb9Xc",
    "bJ_7nYEpkBo",
    "O9D4vnBK2EY",
    "TVK4tHUDw4k",
    "Gn9tDgSYxUw",
    "__BF3XGyC2A",
    "p9CDKtc1Cno",
    "QDfw2ZjtWsQ",
    "ZwAhSaZXq8s",
    "Ryl62yEEo0k",
    "SLBl9O1mfVw",
    "EcpwNuNV81g",
    "hRfw5LiO-II",
    "lKAKcyxNMTo",
    "q5ZBRo_PEGI",
    "_Ke_Sx54nqg",
    "0RvYSvkXJTY",
    "p-gma8xBhtc",
    "jRbgQdMwfIg",
    "vV5FIfYNToI",
    "qo0pS6-_7fU",
    "xYlrGYMsEFQ",
    "3fqrf35BSB4",
    "s5l8U0IuCAs",
    "w2-AYZ-0b0Y",
    "hewWQHInYfE",
    "0zXnPyCKTyM",
    "HVHT1Ylw9dI",
    "G2URlDdDTMo",
    "HK-wBlSUim4",
    "-_lpSR4qkKM",
    "DXH-k1TJKp0",
    "OlOC8Nnpu5U",
    "6S1nuD0qzok",
    "kYsQ5YnlS8s",
    "yPK2PlvnRdw",
    "mVVLjQ-JL3I",
    "1lGct_aPS0Y",
    "ep5wEuJGe54",
    "_Rtey8efdN0",
    "8gvT1tkIM1E",
    "VdYVANxDNkg",
    "Rd8az69QAgk",
    "BD7Jnfziauk",
    "cqh9YVmN-bw",
    "0wnzhf7yQow",
    "0ojDQtWzQ9U",
    "xSTNY3MzePc",
    "jKAXzdkg1ZM",
    "2oXtUngpb-k",
    "a-jG19Ew0Ts",
    "W5grmfiq-3E",
    "Dlx5jjqAbwc",
    "FrPCnqotrKU",
    "-5hhnDy6g_M",
    "2gLsxLymOXU",
    "5ceg8qP2mMk",
    "_AaVUj5_n4Y",
    "w_eHuP-ZjBQ",
    "kElKF-o8yAE",
    "u84pv9IoxLs",
    "MujCppywrSY",
    "VyGPF-7cDSQ",
    "A-qSfOZtkS4",
    "Gbj0jOHoZkw",
    "x_TN6_QQ_hs",
    "mwWGkUChlVs",
    "QDJSAKIE478",
    "7jZ1Jvl7iJ0",
    "KuASnGBsYvs",
    # "7vZ0UcW7uc",
    "L_X-ELsy29A",
    "2iMAncZRopU",
    "OFJUhGC4_hg",
    "fy-WDdCOxtg",
    "so8PplCs_4M",
    "8pw_S-z-NtY",
    "EzzouzKQEC8",
    "P_p7G3NinP8",
    "qPTVVS-E0U4",
    "FIUY2mT5jdw",
    "ETV43779Cho",
    "qwt_5QM2Ncg",
    "0it7r8u4YdM",
    "kBjIGGKq8FI",
    "ODGmMu74pLU",
    "C-JzxdsMi44",
    "Yd3_exRntgI",
    # "diXGcBzb9c",
    "crpFPqIhosI",
    "3o1bj2NlTzU",
    "0iogbdYUI2g",
    "hpXXu8W51Q4",
    "5u8NKsmBpQQ",
    "WAL6UaCfMKo",
    "6xpjCYTRpEg",
    "lXx_9mir89k",
    "aR8s8DayZDg",
    "qgoiowZJJFY",
    "T8B1oHV-2Ig",
    "izVOMmRYdgU",
    "Ok-8pDKxrh4",
    "-6Veo7FBgy0",
    "TX6HpnL0fJ4",
    "BCHuUfO3V_M",
    "ZHWTbE2ZN6U",
    "i0yihJQ27yg",
    "cRhk3U9JDRg",
    "bw_x1oWG3oQ",
    "3LQj4X-JNtk",
    "Vgn6zk4VZEs",
    "zO_5gdnFua0",
    "DRuJTf0Xw9I",
    "FccZzqOMh4A",
    "AueZSrRKILQ",
    "JObMhKnuwV8",
    "Jr9q485COk0",
    "Q25PAh-p9lo",
    "TixL6Tycgho",
    "Vg0pAG0olhk",
    "m-g5kQkqTm8",
    "6koa37cPnAQ",
    "rqg767YC6jM",
    "CrFYHAVppk0",
    "tgt1EXbKbF0",
    "DYlOlv9G3S8",
    "nJAfRMEM4BM",
    "wkS0fPu6mgM",
    "hRdDUGjLVyo",
    "vh5iwTwdrrI",
    "3Lwqu84s5KA",
    "5jcGY7NbaQw",
    "pd7b1Ye6Oc0",
    "4JPAmLuGjxc",
    "ndLLu2dKhVY",
    "J21pL6Ixwoc",
    "0cNbufPYaNQ",
    "d0MS6KKneNE",
    "UVsDDlYw-vE",
    "bZ7xtmkZzrA",
    "EDN8Cm8uTEs",
    "If5ZGXKG9bw",
    "CJqwsIo2NIE",
    "oW5hvQmHLdY",
    "6W9PuLcoZMM",
    "EKdOD12qA6w",
    "DI82cMEhGTI",
    "KIhOgN2gze8",
    "1FFg_iaV_I4",
    "bBulJ5GflP4",
    "IELXveTOkD0",
    "sjFWYuqsmc8",
    "rkTAbWmY-Xc",
    "WVpfJFMiAT8",
    "i3sf67o_Z24",
    "-Wp_0pAf4hM",
    "JRN8CsQBbZk",
    "90vatUNRvBk",
    "fx6whTRhSX4",
    "149RjCpglYI",
    "BkTUWwRZOL8",
    "0to4rm6qkCU",
    "GY7yZiIG-Aw",
    "wy3ugz4S2y0",
    "EemQuYyAMfI",
    "hadkU_g-QZ0",
    "JLKRG_5prAQ",
    "EEZFPg0D_to",
    "WN98EMjmhjU",
    "BbCFDd5y8oA",
    "34IZJuaC6MM",
    "H9bbNJT_RsE",
    "qgGldvMvd7Q",
    "HJHXt_upF1c",
    "vdF63_M-Hjw",
    "bbBZfBRn8NI",
    "-tzvaZ3x9tI",
    "DytRYq0MlCg",
    "1t_dmd1jqKw",
    "-zGH3hD13QQ",
    "ZVl_2e1E8w8",
    "PxgSVxKS69k",
    "Rwx2QYCCwvg",
    "XjwWsuSJrb0",
    "TtzIy1YUVXU",
    "wI_5kbOkTy0",
    "5c-qKZ9Okdw",
    "c-YFPWgvqMw",
    "K3WrKG96FVI",
    "hnVnrn4zVF4",
    "cvzAlmU2EtE",
    "FkOoxzgJ6XM",
    "T8tmW4elTW0",
    "fAJNcskK3Zo",
    "Ve7zunCWxfs",
    "LRiDQy_tq38",
    "LQV4cZMnOwU",
    "jql3d6QjF_M",
    "d8ANsVwTc8w",
    "JLD1IYd0fUA",
    "pcLTW6Rfdh4",
    "-KAemIcX90U",
    "5-CJcXfydV0",
    "0Fsrx_cTBLI",
    "rs8EJNXtyy4",
    "AUtDsHqpd_0",
    "sbtTeP95A3o",
    "WijB-yh_mWU",
    "i0yihJQ27yg",
    "-Wp_0pAf4hM",
    "ZHWTbE2ZN6U",
    "FccZzqOMh4A",
    "90vatUNRvBk",
    "zO_5gdnFua0",
    "KI4hlzHUvZM",
    "3LQj4X-JNtk",
    "bw_x1oWG3oQ",
    "tZRve-Iehsk",
    "DRuJTf0Xw9I",
    "Vgn6zk4VZEs",
    "aR8s8DayZDg",
    "Ok-8pDKxrh4",
    "Q25PAh-p9lo",
    "6xpjCYTRpEg",
    "71Ti_TdpJHE",
    "JObMhKnuwV8",
    "rB0pXI0Fna4",
]

keep_ids = set()
for track_id in track_ids:
  print(track_id)
  try:
    res = Y.yt.get_song(track_id)
    if not res['playabilityStatus']['status'] == 'OK':
      print(f"Error: {res['playabilityStatus']['status']}")
      continue
  except Exception as e:
    print(f"Error: {e}")
    continue
  
  keep_ids.add(track_id)

  
  # break


dUKcpeVvHsU
FKCwRDB42wo
s5Y8o6PKFnc
6rf8QfII7kg
AzRuGcEcaVo
EOkIazVPwEo
vQClDmtYBys
Error: UNPLAYABLE
ac8rL2Ga7pA
PsrFAbFdNWI
jviYDmdLX2A
ayuhW1p_7wo
oYGJVCOklOo
6YFuk97BdPw
xxoO9Da5_Dg
I4ZLZag9GFs
FCvHEXdtPl4
ZG6xibft3pA
1RKlbgvY2gA
r72I3Ps5zYc
UwXA62rDK7U
tMd7qNr5Ct8
Ex0uCvA3tEM
rB0pXI0Fna4
KI4hlzHUvZM
Jl5W8RaG8GM
Uqcu9wZnGL4
_NPUfBpr0RE
2v0yPrdgfRQ
IPcPYsJwJZc
02IkbJkazLs
LfgK9FckrWo
1tE36dBdsP0
Px6unLbNu4Q
05ypnm_Wcls
Wp1hKpIKzNE
Error: UNPLAYABLE
CUIF8cGf3ow
aANSKrs87FA
WmJhMFZEzc8
Error: UNPLAYABLE
HP8b6B9-wNg
vn28G8yLsw
Error: ERROR
TjazJ39k1GY
71Ti_TdpJHE
tZRve-Iehsk
qasPCNQuo88
o8l4LTBwoBE
Ftp4TZKGoug
sk4Js6Q58h8
a5zslcqNCKo
RyZm921d1BU
u-FvKRUTIq4
itiEgttbO0M
-b2XeTb7dY4
mhIQYEoLrjU
AIAw2fAFNMA
Ee7Vg3kf0yY
5SyV43rP-xA
rmJRtSGk-lM
Egfcm2ldXU4
h8R3Oy28BKM
APSCifra1eY
_0gCaSaoULw
_rCswNoJnLI
2seZr4Fl4XA
gIHoUxzRoL8
m09tdbsnxKc
nBGloVKAUyU
2jyGShRb9Xc
bJ_7nYEpkBo
O9D4vnBK2EY
TVK4tHUDw4k
Gn9tDgSYxUw
__BF3XGyC2A
p9CDKtc1Cno
QDfw2ZjtWsQ
ZwAhSaZXq8s
Ryl62yEEo0k
SLBl9O1mfVw
EcpwNuNV81

In [ ]:
print(f"Keep {len(keep_ids)} of {len(track_ids)}")

Keep 267 of 291


In [ ]:
pl_id = Y.yt.create_playlist(
                  title='blues deep rootss', description='llm filter on my shiz',
                  privacy_status='PUBLIC', video_ids=list(keep_ids),
              )

In [ ]:
#blues = Y.playlist_get_info(Y.query_by_title('blues').playlistId)
for track in pl_info['tracks']:
  print(f"Title: {track['title']}\nArtist: "
        f"{track['artists'][0]['name'] if track['artists'] else 'Unknown'}\n" 
        f"Album: {track['album']['name'] if track['album'] else 'Unknown'}\n"
        f"VideoID: {track['videoId']}\n"
  )

Title: Kind-Hearted Blues
Artist: Andrew Hogg
Album: Family Trouble Blues
VideoID: wx6mj_bsjxU

Title: Ain't No Tellin'
Artist: Mississippi John Hurt
Album: American Epic: The Best of Mississippi John Hurt
VideoID: ZG6xibft3pA

Title: I Got a Funny Feeling
Artist: Arthur "Guitar" Kelley
Album: Louisiana Blues
VideoID: 0Fsrx_cTBLI

Title: I'm A Man
Artist: Bo Diddley
Album: Bo Diddley
VideoID: OKlAiuaYdoI

Title: What'd I Say
Artist: Lightnin' Hopkins
Album: The Very Best of Lightnin' Hopkins (Expanded Edition)
VideoID: __BF3XGyC2A

Title: Susie Q
Artist: Sonny Boy Williamson
Album: MOONSHINE
VideoID: uMgskIh2ZV0

Title: Come On In My Kitchen
Artist: Robert Johnson
Album: The Skeleton Key (Original Motion Picture Soundtrack)
VideoID: 2sh483rVV2Y

Title: Cane Break Blues (Instrumental)
Artist: Merle Travis
Album: Walkin' The Strings
VideoID: FXbx1S5sqG0

Title: How Many More Years
Artist: Freddie King
Album: Texas Cannonball
VideoID: zpr0ThSSSOU

Title: Highway 61
Artist: Mississippi Fre

## Upload playlist from tsv backup

In [ ]:
PLAYLIST_NAME = 'zz not like 3'
playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)
# for y in [2020, 2021, 2022, 2023, 2024]:
#   PLAYLIST_NAME = f'y {y} top'
#   playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
#   Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)

### Playlist radio and like mapping helpers

In [ ]:
# Get possible tracks to add to radio like playlist map
import pandas as pd
radio_map = pd.read_csv('..\\playlists\\_ytmusic_radio_to_like_pl_map.tsv', sep='\t')
radio_map = dict(zip(radio_map['radio_playlist'], radio_map['like_playlist']))
# radio_map
all_playlists = frozenset(Y.playlists['title'].unique())
add_to_radio_map = []
for t in sorted(all_playlists):
  if not t.endswith("radio"):
    continue
  if t in radio_map:
    continue  
  if t.replace('radio', 'like') in all_playlists:
   continue
  add_to_radio_map.append(t)
  print(t)


Rock modern The New Age of Classic radio
christmas crooners radio
hiphop get dumb radio
instrumental ambient piano radio
instrumental piano classical radio
punk skate or die radio
xr AtmosphericDnB radio
xr DoomMetal radio
xr Flamenco radio
xr Instrumentals radio
xr ModernRockMusic radio
xr OutlawCountry radio
xr SpaceMusic radio
xr Techno radio
xr TropicalHouse radio
xr acidhouse radio
xr animemusic radio
xr bossanova radio
xr chicagohouse radio
xr melodichouse radio
xs DEZKO ASCEND radio
xs Discord Favorites radio
xs Liquid Drum And Bass radio
zp Dilla radio
['Rock modern The New Age of Classic radio', 'christmas crooners radio', 'hiphop get dumb radio', 'instrumental ambient piano radio', 'instrumental piano classical radio', 'punk skate or die radio', 'xr AtmosphericDnB radio', 'xr DoomMetal radio', 'xr Flamenco radio', 'xr Instrumentals radio', 'xr ModernRockMusic radio', 'xr OutlawCountry radio', 'xr SpaceMusic radio', 'xr Techno radio', 'xr TropicalHouse radio', 'xr acidhouse ra

In [ ]:
# Get possible like playlists, then manually edit and add to _like_playlist.tsv
for _, row in Y.playlists.iterrows():
  t = row['title']
  if not (t.endswith("like") or t.endswith("radio") or t.endswith("albums")):
    print(t)

Liked Music
2022 Recap
2023 Recap
ambiant electro
ambient
ambient BOC
ambient Dream Pop Deep Sleep
ambient haunting harmonious
ambient Indie Synths
ambient japan
beats
beats 2010s wonky LA scene
beats cosmic Slop
beats dj
beats instrumental
Beats Lofi
beats Soulful Instrumentals
bluegrass billy
blues
blues chicago
blues delta roots
blues electric
blues texas roots
Bossa Nova
Brass n chill
Chill Supermix
Chillwave
Discover Weekly
electronic
electronic 1990s electronica
electronic 1990s late 1980s House
electronic 2000s
electronic 2010s
electronic Analog Grooves
electronic big beats
electronic chill
electronic Dance
electronic deep house
electronic dubstep uk
electronic Focus
electronic hard dj
electronic house
electronic house big bass
electronic house dj
electronic house french touch
electronic house funk
electronic House Soulful
electronic House Special
electronic indie essentials
electronic Innerwaves
electronic jaar
electronic new indie beats
electronic soft pad
electronic trance dj

## Query

In [ ]:
Y.query_by_title(playlist_name)

title                                                 jazz radio
playlistId                    PLWptjpDqazOxgrXKM4xfJ-SBRahEDfXmG
thumbnails     [{'url': 'https://yt3.googleusercontent.com/zS...
description                                  Jake G • 797 tracks
count                                                        797
author         [{'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFU...
Name: 20, dtype: object

### Rename Batch Playlists

In [ ]:
VERBOSE = True
DRY_RUN = True

# Rename original subreddit playslists tor shorter no . or _ name
# See: notebooks/old/ytmusic_rename_some_playlists.ipynb
# Example: x_r.bluegrass_tracks_like --> xr bluegrass like
for i, row in Y.playlists.iterrows():
    playlist_name = row['title']
    if playlist_name.startswith("x_r."):
        new_name = playlist_name.replace("x_r.", "xr ").replace("_tracks_", " ").replace("_albums", " albums") 
        if VERBOSE:
            print(f"Renaming: {playlist_name} --> {new_name}")
        if not DRY_RUN:
            Y.rename_playlist(row['playlistId'], new_name)
            
# Manually fix these
Y.find_playlists_with_special_chars(special_chars=["_", "/", ".", ":", ",", "x.r", "x.s" "y_", "x.sp", "Produced", "p_"])

Renaming: x_r.2000smusic_tracks_like --> xr 2000smusic like
Renamed playlist with ID 'PLWptjpDqazOxnmxikFDKOXszYFONGWs8U' to 'xr 2000smusic like'
Renaming: x_r.2000smusic_tracks_radio --> xr 2000smusic radio
Renamed playlist with ID 'PLWptjpDqazOzwVaW_CNc6xtp7v_stZ_dX' to 'xr 2000smusic radio'
Renaming: x_r.2010smusic_tracks_like --> xr 2010smusic like
Renamed playlist with ID 'PLWptjpDqazOwWltLPE8IThgPJeElLQIUO' to 'xr 2010smusic like'
Renaming: x_r.50sMusic_tracks_like --> xr 50sMusic like
Renamed playlist with ID 'PLWptjpDqazOwCLs4ZOeyb_Lg4IgAGlg2L' to 'xr 50sMusic like'
Renaming: x_r.60sMusic_tracks_like --> xr 60sMusic like
Renamed playlist with ID 'PLWptjpDqazOzTAvihnt0PZpLX-bj0JJh2' to 'xr 60sMusic like'
Renaming: x_r.70sMusic_albums --> xr 70sMusic albums
Renamed playlist with ID 'PLWptjpDqazOyo7RbE6Pjv1HJUa61TRhPc' to 'xr 70sMusic albums'
Renaming: x_r.70sMusic_tracks_like --> xr 70sMusic like
Renamed playlist with ID 'PLWptjpDqazOyxy1ElEB7K1qG2A9Pie5ar' to 'xr 70sMusic like'


### Generate playlist for need like from musicbee

In [4]:
uni_module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if uni_module_path not in sys.path: sys.path.append(uni_module_path)
import pandas as pd

DRY_RUN = False
VERBOSE = True
MAX_PL_TRACKS = 4000
PLAYLIST_NAME = 'zz need like'

# This will like them all, need to manually look at result and unlike live or remixes falsly matched
MB_YT_LIKE = os.path.join(uni_module_path, 'tsvs', 'ytmusic_tracks_to_like_from_mb_album_matches.tsv')
mb_yt_like = pd.read_csv(MB_YT_LIKE, sep='\t')
track_ids = mb_yt_like['yt_videoId'].dropna().unique().tolist()
desc = f'{PLAYLIST_NAME} with {len(track_ids)} tracks that are >3.5 in musicbee'
print(f'Accumulated {len(track_ids)} track ids')
if not DRY_RUN:
  print('manually check these, most are false positives, some need like')
  pl_id = Y.yt.create_playlist(
                    title=PLAYLIST_NAME, description='',
                    privacy_status='PRIVATE', video_ids=list(track_ids)
                )
  # pl_id = Y.playlist_from_yt_vids(track_ids, pl_name=PLAYLIST_NAME, sleep=0.5, public='PRIVATE', desc=desc,
  #                         dry=DRY_RUN, set_rating=None, remove_dupes=False, verbose=VERBOSE)
  print(f'Generated playlist: {PLAYLIST_NAME} with id {pl_id}')
  
## TODO 400 err, meant to use for >4000 track max
# if not DRY_RUN:
#   for i in range((len(track_ids) + MAX_PL_TRACKS - 1) // MAX_PL_TRACKS):
#       # chunk_name = f"{PLAYLIST_NAME} {i+1}" if len(track_ids) > MAX_PL_TRACKS else PLAYLIST_NAME
#       vids = track_ids[i*MAX_PL_TRACKS:(i+1)*MAX_PL_TRACKS]
#       pl_id = Y.playlist_from_yt_vids(vids, pl_name=chunk_name, sleep=0.5, public='PRIVATE', desc=desc,
#                               dry=DRY_RUN, set_rating=None, remove_dupes=False, verbose=VERBOSE)
#       print(f'Generated playlist: {chunk_name} with id {pl_id}')

Accumulated 1533 track ids
manually check these, most are false positives, some need like
Generated playlist: zz need like with id PLWptjpDqazOzc8sZVQpg4YBmWKc2uX1J1


## Like all tracks in playlist

Note good to manually check first

In [ ]:
# Initialize the YTMusicPlaylists object

# 1. Find the specific playlist
target_playlist_name = "zz need like"
playlist_row = Y.query_by_title(target_playlist_name)
# 2. Get the full playlist info (including the track list)
pl_info = Y.playlist_get_info(playlist_row['playlistId'], use_cache=False)
# 3. Rate all songs in the playlist as 'LIKE'
results = Y.playlist_rate_all_songs(pl_info, rating='LIKE', verbose=False)
print(f"\nFinished! Results for '{target_playlist_name}':")


### Generate playlist froma a list of albums

In [ ]:
# ORIG
# name = 'y_2023_albums_to_listen_to_v3'
# desc = 'manually selected albums to top off 2023 albums'
# albums_to_add = [
# "Aesop Rock - Integrated Tech Solutions",
# "Mitski - The Land Is Inhospitable and So Are We",
# "Olivia Rodrigo - GUTS",
# "The National - First Two Pages Of Frankenstein",
# "Foo Fighters - But Here We Are",
# "Jessie Ware - That! Feels Good!",
# ]


# TODO run this before playlistbackup
# Playlist for all mb library albums
dry_run = False
verbose = False
uni_module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if uni_module_path not in sys.path: sys.path.append(uni_module_path)
import unify_lib as uni
import pandas as pd
import time

YT_ALBUM_MATCH_TSV = os.path.join(uni_module_path, 'tsvs', 'ytmusic_musicbee_album_matches.tsv')

DATE = time.strftime('%m-%d-%Y')
MAX_PL_TRACKS = 4000

# Just do library
MB_LIB = os.path.join(uni_module_path, 'db_assets', 'musicbee_library.tsv')

# MB_INBOX = os.path.join(uni_module_path, 'db_assets', 'musicbee_inbox.tsv')
name = 'za mb library unmatched albums v2'
desc = f' albums in mb library but not not in ytmusic_musicbee_album_matches.tsv {DATE}'


print('Loading ytmusic and musicbee library track databases (takes ~1m)')
# mb_tracks = uni.ingest_musicbee_db_assets(MB_LIB, MB_INBOX)
mb_tracks = pd.read_csv(MB_LIB, sep='\t')
mb_tracks['fuzzy_album_id'] = mb_tracks.apply(uni.make_mb_fuzzy_album_id, axis=1)
albums_to_add = frozenset(mb_tracks['fuzzy_album_id'].unique())
initial_num_albums = len(albums_to_add)  # Calculate BEFORE removing any albums

last_album_matches = pd.read_csv(YT_ALBUM_MATCH_TSV, sep='\t', index_col=0, low_memory=False)
last_album_matches = last_album_matches.loc[last_album_matches['mb_match_label'] == 'MATCH']
last_album_matches = frozenset(last_album_matches['mb_match'].unique())

albums_removed = albums_to_add.intersection(last_album_matches)
num_removed = len(albums_removed)
albums_to_add -= last_album_matches 
print(f'Loaded {initial_num_albums} unique musicbee albums')
print(f'Loaded {len(last_album_matches)} unique previous matched mb albums already in yt library')
print(f"Removed {num_removed} albums ({(num_removed / initial_num_albums):.0%}) already in last_album_matches.")
print(f'Attempting to add {len(albums_to_add)} albums to ytmusic playlist')

track_ids = []
skipped = set()
for i, a in enumerate(albums_to_add):
    if a in skipped: continue
    res = Y.yt.search(query=a, filter='albums', limit=1)
    if len(res) == 0 or res[0].get('browseId') == None:
        a2 = a.split(' [')[0].split(' (')[0]
        res = Y.yt.search(query=a2, filter='albums', limit=1)
        if len(res) == 0 or res[0].get('browseId') == None:
          if verbose:
            print(f'Skipping query: {a} empty album match result: {res}')
          skipped.add(a)
          continue
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        t_vid = t['videoId'] 
        # if t_vid not in added_vids:
        track_ids.append(t_vid)
          # print('+', end='')
    if verbose:
      print(f'({i+1}) Added {len(yt_album["tracks"])} tracks":\tq: {a}\tr: {result_album.lower()}')
    if i % 100 == 0:
      print(int(100*i/len(albums_to_add)), end='%...\n')

print(f'Accumulated {len(track_ids)} track ids')

if not dry_run:
  for i in range((len(track_ids) + MAX_PL_TRACKS - 1) // MAX_PL_TRACKS):
      chunk_name = f"{name} {i+1}" if len(track_ids) > MAX_PL_TRACKS else name
      vids = track_ids[i*MAX_PL_TRACKS:(i+1)*MAX_PL_TRACKS]
      pl_id = Y.playlist_from_yt_vids(vids, pl_name=chunk_name, sleep=0.5, public='PRIVATE', desc=desc,
                              dry=dry_run, set_rating=None, remove_dupes=False, verbose=False)
      
      print(f'Generated playlist: {chunk_name} with id {pl_id}')

Loading ytmusic and musicbee library track databases (takes ~1m)


C:\Users\jake\AppData\Local\Temp\ipykernel_13480\1568832776.py:39: DtypeWarning: Columns (7,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  mb_tracks = pd.read_csv(MB_LIB, sep='\t')


Loaded 5532 unique musicbee albums
Loaded 4962 unique previous matched mb albums already in yt library
Removed 4228 albums (76%) already in last_album_matches.
Attempting to add 1304 albums to ytmusic playlist
0%...
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++7%...
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++15%...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++30%...
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++53%...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++61%...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++69%...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++76%...
+++++++

In [22]:
len(added_vids)

14081

### Publicize Like Playlists

In [26]:
# Get possible tracks to add to radio like playlist map
import pandas as pd
from pprint import pprint
import time

DATE = time.strftime('%m-%d-%Y')

like_playlists = frozenset(pd.read_csv('..\\playlists\\_like_playlists.tsv', sep='\t')['title'])
radio_map = pd.read_csv('..\\playlists\\_ytmusic_radio_to_like_pl_map.tsv', sep='\t')
like_playlists_mapped = frozenset(radio_map['like_playlist'])
# radio_map = dict(zip(radio_map['radio_playlist'], radio_map['like_playlist']))


#### Run regularly to update privacy to public

maxes if if too many set to public in one day

In [27]:
dry_run = False
print_warn = False
set_public = True # maxes out quickly
min_count_for_public = 15
finished = []
for i, pl in Y.playlists.iterrows():
  p = pl['title']
  if p not in like_playlists:
    if 'like' in p and  not 'not like' in p:
      print('Skipping not in like_playlist:', p)
    continue
    
  if p not in like_playlists_mapped:
    if print_warn: print('Warning not in like_playlist_mapped:', p)
    pass
  
      
  # tmp reset desc
  desc=None
  if p.startswith('xr '):
    desc = f"favorites compiled from reddit sub r/{p.replace('xr ', '')}".strip()
       
  # set to public if not set and enough tracks
  privacy = None
  if set_public:
    privacy = 'PUBLIC'
    count = int(str(pl['count']).replace(',', ''))
    if count < min_count_for_public:
      print(f'Keeping private too small: {pl["count"]} < {min_count_for_public}:', p)
      privacy = None
    
    metadata = Y.playlist_get_info(pl["playlistId"])
    if metadata['privacy'] == privacy:
      print('Already public:', p)
      privacy = None

  # Apply
  if dry_run:
    print(f"Y.yt.edit_playlist(playlistId={pl['playlistId']}, privacyStatus={privacy}, description={desc})")
  else:
    status = Y.yt.edit_playlist(playlistId=pl['playlistId'],  privacyStatus=privacy, description=desc)
    print(f"Updated {p} with status={status}, privacy={privacy}, description={desc}")
    finished.append(p)


Already public: future beats
Updated future beats with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: reggae classic
Updated reggae classic with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: Reggae
Updated Reggae with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: electronic jaar
Updated electronic jaar with status=STATUS_SUCCEEDED, privacy=None, description=None
Updated rock classic with status=STATUS_SUCCEEDED, privacy=PUBLIC, description=None
Already public: rock 1970s classic
Updated rock 1970s classic with status=STATUS_SUCCEEDED, privacy=None, description=None
Updated xr chillmusic with status=STATUS_SUCCEEDED, privacy=PUBLIC, description=favorites compiled from reddit sub r/chillmusic
Already public: jazz blue note
Updated jazz blue note with status=STATUS_SUCCEEDED, privacy=None, description=None
Already public: reggae covers
Updated reggae covers with status=STATUS_SUCCEEDED, privacy=None, description=

#### (one time) check like playlists, make sure valid

In [ ]:
# dry_run = False
# log_playlist = False
# reset_description = True
# finished = []

# for i, pl in Y.playlists.iterrows():
#   p = pl['title']

#   # Filter 'valid' to like playlist
#   if p in finished:
#     print('Skip already finished:', p)
#     continue
#   if 'not like' in p:
#     # print('Skip:', p)
#     continue
#   if  '_like' in p:
#     print('Skip playlist has "_like":', p)
#     continue
#   if ' like' not in p:
#     continue
#   elif not p.endswith('like'):
#     print('Weird case not end with like:', p)
#     continue
#   if p not in like_playlists:
#     # print(p) # update _like_playlist.tsv
#     print('Skip Not logged as _like_playlist:', p)
#     continue
#   elif p not in like_playlists_mapped:
#     # print(f'{p.replace("like", "radio")}\t{p}') # update radio_to_like_pl_map
#     print('Skip Not logged as radio to like map:', p)
#     continue
#   if len(pl['playlistId']) != 34:
#     print('Skip invalid playlisdId', pl['playlistId'], p)
    
    
#   assert p.endswith(' like')
#   new_p = p.replace(' like', '')
#   if log_playlist:
#     p_dict = pl.to_dict()
#     p_dict.pop('thumbnails')
#     pprint(p_dict, sort_dicts=False)

#   # if not p.startswith('xr'): print(p)
#   desc = None
#   if reset_description:
#     # print('Reset Description for:', p, desc)
#     desc = 'favorites'
#     if p.startswith('xr '):
#       desc += f" compiled from reddit sub r/{p.replace(' xr', '')}"
    
  
#   if dry_run:
#     print(f"Y.yt.edit_playlist(playlistId={pl['playlistId']}, title={new_p}, description={desc})")
#   else:
#     status = Y.yt.edit_playlist(playlistId=pl['playlistId'], title=new_p, description=desc)
#     print(f"Updated {p} with status={status}, reset_description={reset_description}")
#     finished.append(p)
    

Skip already finished: xr PunkRock like
Skip already finished: xr 90sPunk like
Skip already finished: ambient grouper like
Skip already finished: folk country americana like
Skip already finished: rock underground like
Skip already finished: xr deepcuts like
Skip already finished: trip hop & Ambien® like
Skip already finished: xr trapmuzik like
Skip already finished: oldies fallout 40s like
Skip already finished: xr 90sRock like
Skip already finished: xr SoundsVintage like
Skip already finished: xr chillmusic like
Skip already finished: xr OutlawCountry like
Skip already finished: psych ish rock modern like
Skip already finished: rock garage like
Skip already finished: xr treemusic like
Skip already finished: brass like
Skip already finished: xr 70sMusic like
Skip already finished: xr 90sMusic like
Skip already finished: xr DreamPop like
Skip already finished: xr theOverload like
Skip already finished: xr PsychedelicRock like
Skip already finished: xr Rock like
Skip already finished: a